# Multiprocessing in Python

---

## **Introduction to Multiprocessing**

Multiprocessing allows you to **leverage multiple CPU cores** to execute tasks in parallel, improving performance for CPU-bound operations.

**Check the number of available CPU cores:**

In [ ]:
from multiprocessing import cpu_count
print(cpu_count())  # Output: Number of CPU cores (e.g., 8)

---

## **Map-Reduce Example**

### **Sequential Execution**
Consider a function that simulates a time-consuming task, such as computing the square of a number after a delay.

In [ ]:
from time import sleep

def delayed_square(x):
    sleep(1)  # Simulate a delay
    return x * x

data = list(range(8))

**Using a generator expression:**

In [ ]:
%time sum(delayed_square(x) for x in data)

**Output:** ~8 seconds (sequential execution)

**Using `map`:**

In [ ]:
%time sum(map(delayed_square, data))

**Output:** ~8 seconds (still sequential)

---
### **Parallel Execution**
To process each `delayed_square` call **independently and in parallel**, use multiple processes.

---
## **Threads vs. Processes: Key Differences**

| Feature               | **Process**                          | **Thread**                          |
|-----------------------|--------------------------------------|-------------------------------------|
| **Definition**        | Instance of a running program.       | Lightweight unit of execution within a process. |
| **Memory**            | Has its own **memory space**.        | Shares memory with other threads in the same process. |
| **Isolation**         | Self-contained execution environment.| Less isolated; threads share resources. |
| **Communication**     | Requires inter-process communication (IPC). | Easier communication (shared memory). |
| **Overhead**          | Higher overhead (separate memory).   | Lower overhead.                     |
| **Concurrency**       | True parallelism on multi-core systems. | Concurrent on multi-core systems, but limited by Python's GIL. |
| **Use Case**          | CPU-bound tasks.                     | I/O-bound tasks.                    |

**Note:** In Python, **threads are limited by the Global Interpreter Lock (GIL)**, which prevents true parallelism for CPU-bound tasks. Use **processes** for CPU-bound parallelism.

---
## **Multiprocessing vs. Multithreading**

### **Memory Considerations**
- **Processes**: Each process has its own **copy of the data**, which can lead to higher memory usage but better isolation.
- **Threads**: Share the same memory space, which is **memory-efficient** but can lead to race conditions if not managed carefully.

**Best Practice:**
- Use **threads** for I/O-bound tasks (e.g., network requests, file operations).
- Use **processes** for CPU-bound tasks (e.g., mathematical computations, data processing).

---
## **Best Practices for Parallelism in Python**

1. **Profile Your Code**: Identify bottlenecks before parallelizing.
2. **Use `concurrent.futures`**: The `concurrent.futures` module provides a high-level interface for asynchronously executing callables.
   - **`ProcessPoolExecutor`**: Ideal for CPU-bound tasks.
   - **`ThreadPoolExecutor`**: Ideal for I/O-bound tasks.
3. **Avoid Overhead**: Parallelism introduces overhead. Ensure tasks are **large enough** to justify parallel execution.
4. **Describe Tasks as Functions**: Use tools like `map` for parallelism.
5. **Optimize Tasks**: Making individual tasks **faster** is often as important as parallelizing them.

---
## **Example: Parallel Pi Computation**

**Task:** Estimate the value of π using a Monte Carlo simulation in parallel.

### **Sequential Implementation**

In [ ]:
import time
import random

def compute_pi(n):
    count = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x**2 + y**2 <= 1:
            count += 1
    return count

# Sequential execution
elapsed_time = time.time()
nb_simulations = 4
n = 10**7
result = [compute_pi(n) for _ in range(nb_simulations)]
pi = 4 * sum(result) / (n * nb_simulations)
print(f"Estimated value of Pi: {pi:.8f}, Time: {time.time() - elapsed_time:.8f} seconds")

### **Parallel Implementation with `ProcessPoolExecutor`**

In [ ]:
from concurrent.futures import ProcessPoolExecutor

# Parallel execution
elapsed_time = time.time()
with ProcessPoolExecutor() as executor:
    result = list(executor.map(compute_pi, [n] * nb_simulations))
pi = 4 * sum(result) / (n * nb_simulations)
print(f"Estimated value of Pi: {pi:.8f}, Time: {time.time() - elapsed_time:.8f} seconds")

---
## **Exercise: Parallel Pi Computation**

1. **Use Asynchronous Futures**: Implement the Pi computation using `concurrent.futures` with asynchronous futures.
2. **Use Joblib**: Implement the Pi computation using the [`joblib`](https://joblib.readthedocs.io/en/latest/) library.

**Example with Joblib:**

In [ ]:
from joblib import Parallel, delayed

elapsed_time = time.time()
result = Parallel(n_jobs=4)(delayed(compute_pi)(n) for _ in range(nb_simulations))
pi = 4 * sum(result) / (n * nb_simulations)
print(f"Estimated value of Pi: {pi:.8f}, Time: {time.time() - elapsed_time:.8f} seconds")

---
## **Parallel Tools for Python**

The standard library's parallel tools are **limited**. For more powerful features, consider:

- **[Joblib](https://joblib.readthedocs.io/en/latest/)**: Simple helper class for parallel `for` loops using multiprocessing.
- **[Dask](https://dask.org/)**: Parallel computing library that scales from laptops to clusters.
- **[PySpark](https://spark.apache.org/docs/latest/api/python/index.html)**: Distributed computing framework for large-scale data processing.
- **[mpi4py](https://mpi4py.readthedocs.io/)**: Python bindings for the MPI (Message Passing Interface) standard.

---